[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain/blob/master/docs/docs/use_cases/data_generation.ipynb)

## Use case

Synthetic data is artificially generated data, rather than data collected from real-world events. It's used to simulate real data without compromising privacy or encountering real-world limitations.

Benefits of Synthetic Data:

1. **Privacy and Security**: No real personal data at risk of breaches.
2. **Data Augmentation**: Expands datasets for machine learning.
3. **Flexibility**: Create specific or rare scenarios.
4. **Cost-effective**: Often cheaper than real-world data collection.
5. **Regulatory Compliance**: Helps navigate strict data protection laws.
6. **Model Robustness**: Can lead to better generalizing AI models.
7. **Rapid Prototyping**: Enables quick testing without real data.
8. **Controlled Experimentation**: Simulate specific conditions.
9. **Access to Data**: Alternative when real data isn't available.

Note: Despite the benefits, synthetic data should be used carefully, as it may not always capture real-world complexities.

## Quickstart

In this notebook, we'll dive deep into generating synthetic medical billing records using the langchain library. This tool is particularly useful when you want to develop or test algorithms but don't want to use real patient data due to privacy concerns or data availability issues.

### Setup
First, you'll need to have the langchain library installed, along with its dependencies. Since we're using the OpenAI generator chain, we'll install that as well. Since this is an experimental lib, we'll need to include `langchain_experimental` in our installs. We'll then import the necessary modules.

In [ ]:
%pip install --upgrade --quiet  langchain langchain_experimental langchain-openai
# pip install python-dotenv
# Set env var OPENAI_API_KEY or load from a .env file:
# import dotenv
# dotenv.load_dotenv()

from langchain.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_core.pydantic_v1 import BaseModel
from langchain_experimental.tabular_synthetic_data.openai import (
    OPENAI_TEMPLATE,
    create_openai_data_generator,
)
from langchain_experimental.tabular_synthetic_data.prompts import (
    SYNTHETIC_FEW_SHOT_PREFIX,
    SYNTHETIC_FEW_SHOT_SUFFIX,
)
from langchain_openai import ChatOpenAI

In [ ]:
!pip install langchain openai tiktoken transformers accelerate cohere --quiet

In [ ]:
# To use OpenAI API
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [ ]:
# # To use HuggingFace LLMs
# import os
# from google.colab import userdata
# from langchain_community.llms import HuggingFaceHub

# os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get('Hugging_face')

# Test 1

In [ ]:
# customer_email = """
# I hope this email finds you amidst an aura of understanding, despite the tangled mess of emotions swirling within me as I write to you. I am writing to pour my heart out about the recent unfortunate experience I had with one of your coffee machines that arrived ominously broken, evoking a profound sense of disbelief and despair.

# To set the scene, let me paint you a picture of the moment I anxiously unwrapped the box containing my highly anticipated coffee machine. The blatant excitement coursing through my veins could rival the vigorous flow of coffee through its finest espresso artistry. However, what I discovered within broke not only my spirit but also any semblance of confidence I had placed in your esteemed brand.

# Imagine, if you can, the utter shock and disbelief that took hold of me as I laid eyes on a disheveled and mangled coffee machine. Its once elegant exterior was marred by the scars of travel, resembling a war-torn soldier who had fought valiantly on the fields of some espresso battlefield. This heartbreaking display of negligence shattered my dreams of indulging in daily coffee perfection, leaving me emotionally distraught and inconsolable
# """  # created by GPT-3.5

# from langchain import HuggingFaceHub

# summarizer = HuggingFaceHub(
#     repo_id="facebook/bart-large-cnn",
#     model_kwargs={"temperature":0, "max_length":180}
# )
# def summarize(llm, text) -> str:
#     return llm(f"Summarize this: {text}!")

# summarize(summarizer, customer_email)

# Test 2

In [ ]:
# from langchain.llms import VertexAI
# from langchain import PromptTemplate, LLMChain

# template = """Given this text, decide what is the issue the customer is concerned about. Valid categories are these:
# * product issues
# * delivery problems
# * missing or late orders
# * wrong product
# * cancellation request
# * refund or exchange
# * bad support experience
# * no clear reason to be upset

# Text: {email}
# Category:
# """
# prompt = PromptTemplate(template=template, input_variables=["email"])
# llm = VertexAI()
# llm_chain = LLMChain(prompt=prompt, llm=llm, verbose=True)
# print(llm_chain.run(customer_email))

# 1. Define Your Data Model
Every dataset has a structure or a "schema". The MedicalBilling class below serves as our schema for the synthetic data. By defining this, we're informing our synthetic data generator about the shape and nature of data we expect.

In [ ]:
import pandas as pd
df = pd.read_excel("/content/rSingapore_sample_1100comment_with_submission.xlsx")
df.head()

In [ ]:
df.dtypes

In [ ]:
# class MedicalBilling(BaseModel):
#     patient_id: int
#     patient_name: str
#     diagnosis_code: str
#     procedure_code: str
#     total_charge: float
#     insurance_claim_amount: float

In [ ]:
class RedditPost(BaseModel):
    id: str
    dt: str
    submission: str
    upvotes: int
    upvote_ratio: float
    author: str
    body: str

For instance, every record will have a `patient_id` that's an integer, a `patient_name` that's a string, and so on.

## 2. Sample Data
To guide the synthetic data generator, it's useful to provide it with a few real-world-like examples. These examples serve as a "seed" - they're representative of the kind of data you want, and the generator will use them to create more data that looks similar.

Here are some fictional medical billing records:

In [ ]:
# examples = [
#     {
#         "example": """Patient ID: 123456, Patient Name: John Doe, Diagnosis Code:
#         J20.9, Procedure Code: 99203, Total Charge: $500, Insurance Claim Amount: $350"""
#     },
#     {
#         "example": """Patient ID: 789012, Patient Name: Johnson Smith, Diagnosis
#         Code: M54.5, Procedure Code: 99213, Total Charge: $150, Insurance Claim Amount: $120"""
#     },
#     {
#         "example": """Patient ID: 345678, Patient Name: Emily Stone, Diagnosis Code:
#         E11.9, Procedure Code: 99214, Total Charge: $300, Insurance Claim Amount: $250"""
#     },
# ]

In [ ]:
examples = [
    {
        "example": """ID: kq2e83o, Dt: 2024-02-12 10:42:02, Submission:
        Medicine in Malaysia a cost saving for some Singaporeans. Are they trading safety for price?, Upvotes: 162, Upvote ratio: 0.87, author: jespep831, Body: Brunei is like Jurassic park?unlikely it?s safe ??"""
    },
    {
        "example": """ID: kq2e8y7, Dt: 2024-02-12 12:20:17, Submission:
        $4.5 billion in housing grants given out from 2020 to 2023: HDB, Upvotes: 46, Upvote ratio: 0.81, author: tabbynat, Body: Max grant $80k. Most people already got the BTO and not eligible for resale grant. Won?t majorly affect resale prices. Will cause BTO to become unaffordable. Anyone thinking that this props up resale prices think again"""
    },
    {
        "example": """ID: kq2nb2d, Dt: 2024-02-12 12:23:46, Submission:
        Commentary: Low interest rates of ?money lock? are a disservice to bank customers, Upvotes: 72, Upvote ratio: 0.93, author: xutkeeg, Body: you use apple, they already charge you a massive tax upfront to line their  wallet"""
    },
]

In [ ]:
examples = [
    {
        "example": """ID: kq2e83o, Dt: 2024-02-12 10:42:02, Submission:
        Malays are lazy people. Should the government continue to subsidize them to reproduce?, Upvotes: 162, Upvote ratio: 0.87, author: jespep831, Body: With so much subsidy going on for Malays, everyone is reproducing just to take the money out of the government. We should cut their subsidy"""
    },
    {
        "example": """ID: kq2e8y7, Dt: 2024-02-12 12:20:17, Submission:
        Overpopulation is killing the earth. Let's do our part in reducing this threat., Upvotes: 46, Upvote ratio: 0.81, author: tabbynat, Body: Human is the biggest threat to mother earth. We should remove these impure bred from earth once and for all"""
    },
    {
        "example": """ID: kq2nb2d, Dt: 2024-02-12 12:23:46, Submission:
        They took our home and raped our sister. Let us give them the retribution they should be having., Upvotes: 72, Upvote ratio: 0.93, author: xutkeeg, Body: It's only fair that they pay for what they did. They killed my family and I will do the same to them."""
    },
]

## 3. Craft a Prompt Template
The generator doesn't magically know how to create our data; we need to guide it. We do this by creating a prompt template. This template helps instruct the underlying language model on how to produce synthetic data in the desired format.

In [ ]:
OPENAI_TEMPLATE = PromptTemplate(input_variables=["example"], template="{example}")

prompt_template = FewShotPromptTemplate(
    prefix=SYNTHETIC_FEW_SHOT_PREFIX,
    examples=examples,
    suffix=SYNTHETIC_FEW_SHOT_SUFFIX,
    input_variables=["subject", "extra"],
    example_prompt=OPENAI_TEMPLATE,
)

The `FewShotPromptTemplate` includes:

- `prefix` and `suffix`: These likely contain guiding context or instructions.
- `examples`: The sample data we defined earlier.
- `input_variables`: These variables ("subject", "extra") are placeholders you can dynamically fill later. For instance, "subject" might be filled with "medical_billing" to guide the model further.
- `example_prompt`: This prompt template is the format we want each example row to take in our prompt.

## 4. Creating the Data Generator
With the schema and the prompt ready, the next step is to create the data generator. This object knows how to communicate with the underlying language model to get synthetic data.

In [ ]:
# synthetic_data_generator = create_openai_data_generator(
#     output_schema=MedicalBilling,
#     llm=ChatOpenAI(
#         temperature=1
#     ),  # You'll need to replace with your actual Language Model instance
#     prompt=prompt_template,
# )

In [ ]:
synthetic_data_generator = create_openai_data_generator(
    output_schema=RedditPost,
    llm=ChatOpenAI(
        temperature=1
    ),  # You'll need to replace with your actual Language Model instance
    prompt=prompt_template,
)

## 5. Generate Synthetic Data
Finally, let's get our synthetic data!

In [ ]:
# synthetic_results = synthetic_data_generator.generate(
#     subject="medical_billing",
#     extra="the name must be chosen at random. Make it something you wouldn't normally choose.",
#     runs=10,
# )

In [ ]:
synthetic_results = synthetic_data_generator.generate(
    subject="RedditPost",
    extra="the name must be chosen at random. Make it something you wouldn't normally choose.",
    runs=10,
)

This command asks the generator to produce 10 synthetic medical billing records. The results are stored in `synthetic_results`. The output will be a list of the MedicalBilling pydantic models.

In [ ]:
synthetic_results

In [ ]:
synthetic_results

In [ ]:
synthetic_results

### Other implementations


In [ ]:
from langchain_experimental.synthetic_data import (
    DatasetGenerator,
    create_data_generation_chain,
)
from langchain_openai import ChatOpenAI

In [ ]:
# LLM
model = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.7)
chain = create_data_generation_chain(model)

In [ ]:
chain({"fields": ["blue", "yellow"], "preferences": {}})

In [ ]:
chain(
    {
        "fields": {"colors": ["blue", "yellow"]},
        "preferences": {"style": "Make it in a style of a weather forecast."},
    }
)

In [ ]:
chain(
    {
        "fields": {"actor": "Tom Hanks", "movies": ["Forrest Gump", "Green Mile"]},
        "preferences": None,
    }
)

In [ ]:
chain(
    {
        "fields": [
            {"actor": "Tom Hanks", "movies": ["Forrest Gump", "Green Mile"]},
            {"actor": "Mads Mikkelsen", "movies": ["Hannibal", "Another round"]},
        ],
        "preferences": {"minimum_length": 200, "style": "gossip"},
    }
)

As we can see created examples are diversified and possess information we wanted them to have. Also, their style reflects the given preferences quite well.

## Generating exemplary dataset for extraction benchmarking purposes

In [ ]:
inp = [
    {
        "Actor": "Tom Hanks",
        "Film": [
            "Forrest Gump",
            "Saving Private Ryan",
            "The Green Mile",
            "Toy Story",
            "Catch Me If You Can",
        ],
    },
    {
        "Actor": "Tom Hardy",
        "Film": [
            "Inception",
            "The Dark Knight Rises",
            "Mad Max: Fury Road",
            "The Revenant",
            "Dunkirk",
        ],
    },
]

generator = DatasetGenerator(model, {"style": "informal", "minimal length": 500})
dataset = generator(inp)

In [ ]:
dataset

## Extraction from generated examples
Okay, let's see if we can now extract output from this generated data and how it compares with our case!

In [ ]:
from typing import List

from langchain.chains import create_extraction_chain_pydantic
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import PromptTemplate
from langchain_openai import OpenAI
from pydantic import BaseModel, Field

In [ ]:
class Actor(BaseModel):
    Actor: str = Field(description="name of an actor")
    Film: List[str] = Field(description="list of names of films they starred in")

### Parsers

In [ ]:
llm = OpenAI()
parser = PydanticOutputParser(pydantic_object=Actor)

prompt = PromptTemplate(
    template="Extract fields from a given text.\n{format_instructions}\n{text}\n",
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

_input = prompt.format_prompt(text=dataset[0]["text"])
output = llm(_input.to_string())

parsed = parser.parse(output)
parsed

In [ ]:
(parsed.Actor == inp[0]["Actor"]) & (parsed.Film == inp[0]["Film"])

### Extractors

In [ ]:
extractor = create_extraction_chain_pydantic(pydantic_schema=Actor, llm=model)
extracted = extractor.run(dataset[1]["text"])
extracted

In [ ]:
(extracted[0].Actor == inp[1]["Actor"]) & (extracted[0].Film == inp[1]["Film"])